In [55]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [56]:
def impurity(y, mode):
    p = np.unique(y, return_counts=True)[1] / len(y)
    return -np.sum(p*np.log2(p+1e-9)) if mode=="entropy" else 1-np.sum(p**2)


def split(y, X_col, t):
    return y[X_col<=t], y[X_col>t]


def information_gain(X_col, y, t):
    L, R = split(y, X_col, t)
    if len(L)==0 or len(R)==0: return 0
    n = len(y)
    return impurity(y,"entropy") - ((len(L)/n)*impurity(L,"entropy") + (len(R)/n)*impurity(R,"entropy"))


def gain_ratio(X_col, y, t):
    L, R = split(y, X_col, t)
    if len(L)==0 or len(R)==0: return 0
    n = len(y)
    ig = information_gain(X_col, y, t)

    si = sum([-(len(s)/n)*np.log2(len(s)/n + 1e-9) for s in [L,R]])
    return ig/si if si!=0 else 0


def gini_gain(X_col, y, t):
    L, R = split(y, X_col, t)
    if len(L)==0 or len(R)==0: return 0
    n = len(y)
    return impurity(y,"gini") - ((len(L)/n)*impurity(L,"gini") + (len(R)/n)*impurity(R,"gini"))


In [57]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


In [58]:
class DecisionTree:
    def __init__(self, criterion):
        self.criterion = criterion
        self.root = None

    # Choose correct gain function
    def _gain(self, X_col, y, t):
        if self.criterion == "id3":
            return information_gain(X_col, y, t)
        elif self.criterion == "c45":
            return gain_ratio(X_col, y, t)
        return gini_gain(X_col, y, t)

    # Find best feature + threshold
    def best_split(self, X, y):
        best_gain, best_feature, best_thresh = -1, None, None

        for f in range(X.shape[1]):
            for t in np.unique(X[:, f]):
                gain = self._gain(X[:, f], y, t)

                if gain > best_gain:
                    best_gain, best_feature, best_thresh = gain, f, t

        return best_feature, best_thresh

    # Build tree recursively
    def build_tree(self, X, y):

        # Stop if all labels same
        if len(np.unique(y)) == 1:
            return Node(value=y[0])

        feature, thresh = self.best_split(X, y)

        # Stop if no split found
        if feature is None:
            return Node(value=np.bincount(y).argmax())

        left_mask = X[:, feature] <= thresh
        right_mask = X[:, feature] > thresh

        left = self.build_tree(X[left_mask], y[left_mask])
        right = self.build_tree(X[right_mask], y[right_mask])

        return Node(feature, thresh, left, right)

    def fit(self, X, y):
        self.root = self.build_tree(X, y)

    # Predict for single sample
    def _predict(self, x, node):
        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self._predict(x, node.left)
        return self._predict(x, node.right)

    # Predict for dataset
    def predict(self, X):
        return np.array([self._predict(x, self.root) for x in X])


In [59]:
datasets = {
    "Iris": load_iris(),
    "Wine": load_wine(),
    "Breast Cancer": load_breast_cancer()
}


In [60]:
results = []

for name, data in datasets.items():
    X = data.data
    y = data.target

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    for algo in ["id3", "c45", "cart"]:
        tree = DecisionTree(algo)
        tree.fit(X_train, y_train)
        preds = tree.predict(X_test)

        results.append([
            name,
            algo.upper(),
            accuracy_score(y_test, preds),
            precision_score(y_test, preds, average="macro"),
            recall_score(y_test, preds, average="macro"),
            f1_score(y_test, preds, average="macro")
        ])


In [61]:
comparison = pd.DataFrame(
    results,
    columns=["Dataset", "Algorithm", "Accuracy", "Precision", "Recall", "F1-Score"]
)

comparison


,Dataset,Algorithm,Accuracy,Precision,Recall,F1-Score
0,Iris,ID3,0.911111,0.897436,0.897436,0.897436
1,Iris,C45,0.955556,0.955556,0.948718,0.948413
2,Iris,CART,0.955556,0.955556,0.948718,0.948413
3,Wine,ID3,0.851852,0.853024,0.847536,0.847621
4,Wine,C45,0.888889,0.878571,0.880952,0.879374
5,Wine,CART,0.944444,0.950794,0.942774,0.946140
6,Breast Cancer,ID3,0.935673,0.932154,0.929233,0.930654
7,Breast Cancer,C45,0.923977,0.915584,0.923280,0.919096
8,Breast Cancer,CART,0.923977,0.914406,0.926587,0.919576
